In [4]:
import pandas as pd
import numpy as np
from EssSimulation_withoutMaxDemand import EssSimulationModel
import calendar
import copy
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif']=['SimHei']    # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False    # 用来显示负号

In [5]:
exp_name = "estimate1122"
node_name = "route_10"
es_scale = 625
time_ratio = 15 / 60

In [6]:
es_info = {"transform_capacity": 8883000,
            "invertband": 0,
            "soc_redundant_ratio": 0,
            "usable_depth": 0.90,
            "charge_loss": 0.92,
            "discharge_loss": 0.95,
            "es_charge_max": es_scale,
            "es_charge_min": -es_scale,
            "es_capacity_max": 1305, #es_scale * 2
            "es_capacity_min": 0}

In [7]:
def split_load_by_month(df):
    """
    将以时间戳为索引、包含'value'列的DataFrame按月拆分，返回一个字典。
    
    参数:
        df (pd.DataFrame): 索引为时间对象（datetime-like），包含'value'列。
    
    返回:
        dict: 键为'YYYY-MM'格式的字符串，值为对应月份的'value' Series。
    """
    # 确保索引是 datetime 类型
    if not isinstance(df.index, pd.DatetimeIndex):
        df = df.copy()
        df.index = pd.to_datetime(df.index)
    
    # 按月分组
    monthly_groups = df.groupby(df.index.to_period('M'))
    
    # 构建字典：key 为 'YYYY-MM' 字符串，value 为该月的 'value' Series
    result = {
        str(period): group
        for period, group in monthly_groups
    }
    
    return result

def get_days_in_month(date_str):
    try:
        year, month = map(int, date_str.split('-'))
        # calendar.monthrange(year, month) 返回 (weekday_of_first_day, number_of_days)
        _, days = calendar.monthrange(year, month)
        return days
    except ValueError as e:
        print(f"输入格式错误或无效日期: {e}")
        return None

In [8]:
demand_load_df = pd.read_csv(f"./data/{exp_name}/{node_name}/demand_load.csv")
demand_load_df['time'] = pd.to_datetime(demand_load_df['time'])
demand_load_df.set_index('time', inplace=True)

strategy_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/es_scale_experiment/schedule_result_scale_{es_scale}.csv")
strategy_df.rename(columns={"power_opt": "value"}, inplace=True)
strategy_df['time'] = pd.to_datetime(strategy_df['time'])
strategy_df.set_index('time', inplace=True)

ele_price_df = pd.read_csv(f"./data/{exp_name}/{node_name}/ele_price.csv")
ele_price_df['time'] = pd.to_datetime(ele_price_df['time'])
ele_price_df.set_index('time', inplace=True)

In [6]:
simulation_model = EssSimulationModel(es_info)
es_charge_df, es_soc_df, total_load_df = simulation_model.simulation_process(demand_load_df, strategy_df, 0)

In [7]:
month_load_dict = split_load_by_month(es_charge_df)

In [8]:
equivalent_charge_list = []
equivalent_discharge_list = []
result_df = pd.DataFrame(columns=[
    "实际放电小时数",
    "实际充电小时数",
    "等效放电小时数",
    "等效充电小时数"
])
for k,v in month_load_dict.items():
    days = get_days_in_month(k)
    count1 = (v['value'] > 0).sum() * time_ratio / days
    count2 = (v['value'] < 0).sum() * time_ratio / days
    count3 = v.loc[v['value'] > 0, 'value'].sum() * time_ratio / es_scale / days
    count4 = -v.loc[v['value'] < 0, 'value'].sum() * time_ratio / es_scale / days
    result_df.loc[k] = [count1, count2, count3, count4]
    equivalent_discharge_list.append(count3 / 4 * days)
    equivalent_charge_list.append(count4 / 4 * days)
equivalent_charge_days = sum(equivalent_charge_list)
equivalent_discharge_days = sum(equivalent_discharge_list)
print(f"年等效充电天数:{equivalent_charge_days}, 年等效放电天数:{equivalent_discharge_days}",)

年等效充电天数:324.1284805139588, 年等效放电天数:283.2882919691999


In [9]:
result_df.to_csv("hour_result_625.csv", encoding="utf_8_sig")

In [2]:
len(list(range(5000, 28000, 500)))

46

In [9]:
demand_load_df['value'].resample('M').max()

C:\Users\hobo\AppData\Local\Temp\ipykernel_27252\4148266694.py:1: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  demand_load_df['value'].resample('M').max()


time
2024-07-31    8361.6
2024-08-31    8179.2
2024-09-30    8167.2
2024-10-31    7780.8
2024-11-30    7558.8
2024-12-31    7504.8
2025-01-31    7718.4
2025-02-28    7729.2
2025-03-31    7872.0
2025-04-30    7947.6
2025-05-31    8298.0
2025-06-30    8359.2
Freq: ME, Name: value, dtype: float64

In [10]:
demand_load_df['value'].resample('ME').max()

time
2024-07-31    8361.6
2024-08-31    8179.2
2024-09-30    8167.2
2024-10-31    7780.8
2024-11-30    7558.8
2024-12-31    7504.8
2025-01-31    7718.4
2025-02-28    7729.2
2025-03-31    7872.0
2025-04-30    7947.6
2025-05-31    8298.0
2025-06-30    8359.2
Freq: ME, Name: value, dtype: float64

In [11]:
pd.__version__

'2.3.0'